In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
from torch.nn.functional import softmax
import numpy as np

# Load your fine-tuned model
model_path = "../roberta-fake-news-best-local"          # ← change this to your actual folder path
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Optional: move to GPU if available
device = 0 if torch.cuda.is_available() else -1
model.to(device)

# Create a pipeline (easiest way to get probabilities)
pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=True   # important → gives probs for both classes
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [11]:
import json

# Load your full dataset (same file you trained on)
with open("H:\\AIML_project\\Model_desgin\\cleaning_review_samples.json", "r", encoding="utf-8") as f:   # ← your cleaned file name
    data = json.load(f)

# Extract test examples
test_examples = data.get("test", [])   # list of dicts: [{"text": "...", "label": 0 or 1}, ...]

# Convert to simple lists
test_texts = [ex["text"] for ex in test_examples]
test_labels = [ex["label"] for ex in test_examples]

print(test_texts)    
print(f"Loaded {len(test_texts)} test examples")
print(f"Example text: {test_texts[0][:100]}... with label {test_labels[0]}")

["Chicago Police Try to Predict Who May Shoot or Be Shot In a city of 2.7 million people, about 1,400 are responsible for much of the violence, Mr. Johnson said, and all of them are on what the department calls its Strategic Subject List.\n\nSo far this year, more than 70 percent of the people who have been shot in Chicago were on the list, according to the police, as were more than 80 percent of those arrested in connection with shootings.\n\nIn a broad drug and gang raid carried out last week amid a disturbing uptick this year in shootings and murders, the Police Department said 117 of the 140 people arrested were on the list.\n\nAnd in one recent report on homicides and shootings over a two-day stretch, nearly everyone involved was on the list.\n\n“We are targeting the correct individuals,” Mr. Johnson said. “We just need our judicial partners and our state legislators to hold these people accountable.”\n\nMany government agencies and private entities are using data to try to predic

In [9]:
print("Number of examples:", len(test_examples))
print("Types in first few:", [type(x) for x in test_examples[:6]])

short_or_empty = []
for i, item in enumerate(test_examples):
    if not isinstance(item, str):
        print(f"Item {i} is not a string: {type(item)} → {repr(item)}")
        continue
    cleaned = item.strip()
    length = len(cleaned)
    if length < 30:
        short_or_empty.append((i, length, repr(cleaned[:100])))
        
print(f"\nFound {len(short_or_empty)} short or empty texts:")
for i, length, preview in short_or_empty[:10]:  # show first 10
    print(f"  {i:3d} | len={length:3d} | {preview}")

Number of examples: 6
Types in first few: [<class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>]
Item 0 is not a string: <class 'dict'> → {'text': "Chicago Police Try to Predict Who May Shoot or Be Shot In a city of 2.7 million people, about 1,400 are responsible for much of the violence, Mr. Johnson said, and all of them are on what the department calls its Strategic Subject List.\n\nSo far this year, more than 70 percent of the people who have been shot in Chicago were on the list, according to the police, as were more than 80 percent of those arrested in connection with shootings.\n\nIn a broad drug and gang raid carried out last week amid a disturbing uptick this year in shootings and murders, the Police Department said 117 of the 140 people arrested were on the list.\n\nAnd in one recent report on homicides and shootings over a two-day stretch, nearly everyone involved was on the list.\n\n“We are targeting the correct individuals,” Mr. Joh

In [16]:
# Assuming you already have:
# test_texts   = [ex["text"] for ex in test_examples]    # list of strings
# test_labels  = [ex["label"] for ex in test_examples]   # list of 0 or 1

# Quick sanity check
print("Number of examples:", len(test_texts))
print("Length of test_labels:", len(test_labels))
if len(test_texts) > 0:
    print("First text preview:", repr(test_texts[0][:150]))
    print("First label:", test_labels[0])

# Now the inference loop
results = []

for i, text in enumerate(test_texts):
    if i % 1000 == 0:
        print(f"Processing {i}/{len(test_texts)}...")
    
    # Minimal safety check
    if not isinstance(text, str) or len(text.strip()) < 20:
        print(f"Skipping example {i}: too short or not string (len={len(text.strip() if isinstance(text, str) else 0)})")
        continue
    
    try:
        out = pipe(text, truncation=True, max_length=512)
        probs = {d['label']: d['score'] for d in out}
        conf = max(probs.values())
        pred_label = 'real' if max(probs, key=probs.get) == 'LABEL_0' else 'fake'
        true_label = 'real' if test_labels[i] == 0 else 'fake'
        
        results.append({
            'index': i,
            'text_preview': text[:350] + "..." if len(text) > 350 else text,
            'pred': pred_label,
            'true': true_label,
            'confidence': conf
        })
    except Exception as e:
        print(f"Error on example {i}: {e}")
        continue

# ──────────────────────────────────────────────────────────────
# Show results
import numpy as np

if not results:
    print("\nNo examples were processed successfully.")
else:
    conf_array = np.array([r['confidence'] for r in results])
    indices = np.argsort(conf_array)[::-1]  # descending confidence
    
    print(f"\nProcessed {len(results)} / {len(test_texts)} examples")
    print("\nTop confident CORRECT predictions:")
    count = 0
    for idx in indices:
        r = results[idx]
        if r['pred'] == r['true']:
            print(f"{count+1}. Conf: {float(r['confidence']):.4f} | {r['pred'].upper()} (correct) | Text: {r['text_preview']}\n")
            count += 1
            if count >= 10:  # show top 10 correct ones
                break
    
    if count == 0:
        print("No correct high-confidence predictions found.")

Number of examples: 6
Length of test_labels: 6
First text preview: 'Chicago Police Try to Predict Who May Shoot or Be Shot In a city of 2.7 million people, about 1,400 are responsible for much of the violence, Mr. John'
First label: 0
Processing 0/6...

Processed 6 / 6 examples

Top confident CORRECT predictions:
1. Conf: 1.0000 | REAL (correct) | Text: Dean Walks a Tightrope Over Positions on Gun Control This turn of events captures the difficulty that gun control poses for Dr. Dean and the other Democratic contenders as they struggle with an issue that many say helped cost their party the White House and a handful of Congressional seats in 2000.

Polls show overwhelming support among Democratic ...

2. Conf: 1.0000 | REAL (correct) | Text: (Reuters) - Puerto Rico’s governor said on Saturday he has delivered a revised fiscal turnaround plan to the U.S. territory’s financial oversight board that includes $262 million in additional revenue and changes to healthcare funding.  Governor Ri

In [ ]:
print("Number of examples:", len(test_examples))
print("Types in first few:", [type(x) for x in test_examples[:6]])

short_or_empty = []
for i, item in enumerate(test_examples):
    if not isinstance(item, str):
        print(f"Item {i} is not a string: {type(item)} → {repr(item)}")
        continue
    cleaned = item.strip()
    length = len(cleaned)
    if length < 30:
        short_or_empty.append((i, length, repr(cleaned[:100])))
        
print(f"\nFound {len(short_or_empty)} short or empty texts:")
for i, length, preview in short_or_empty[:10]:  # show first 10
    print(f"  {i:3d} | len={length:3d} | {preview}")

Number of examples: 6
Types in first few: [<class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>, <class 'dict'>]
Item 0 is not a string: <class 'dict'> → {'text': "Chicago Police Try to Predict Who May Shoot or Be Shot In a city of 2.7 million people, about 1,400 are responsible for much of the violence, Mr. Johnson said, and all of them are on what the department calls its Strategic Subject List.\n\nSo far this year, more than 70 percent of the people who have been shot in Chicago were on the list, according to the police, as were more than 80 percent of those arrested in connection with shootings.\n\nIn a broad drug and gang raid carried out last week amid a disturbing uptick this year in shootings and murders, the Police Department said 117 of the 140 people arrested were on the list.\n\nAnd in one recent report on homicides and shootings over a two-day stretch, nearly everyone involved was on the list.\n\n“We are targeting the correct individuals,” Mr. Joh

In [33]:
# Cell 3 – Prediction function
LABEL_MAP = {0: "REAL", 1: "FAKE"}
def predict_news(text: str, true_label=None):
    """
    Predict REAL / FAKE for given text.
    Optionally show if prediction matches known label.
    """
    if not text or not text.strip():
        return {"error": "Empty input"}

    try:
        result = pipe(text, truncation=True, max_length=512)
        
        # Handle if result is a single dict (not a list)
        if isinstance(result, dict):
            result = [result]
        
        if not result or len(result) == 0:
            return {"error": "Pipeline returned empty result"}
        
        probs = {d['label']: d['score'] for d in result}
        
        # If we only got 1 score, infer the other as 1 - score
        if len(probs) == 1:
            label, score = list(probs.items())[0]
            other_label = 'LABEL_0' if label == 'LABEL_1' else 'LABEL_1'
            probs[other_label] = 1.0 - score
        
        scores = list(probs.values())
        prob_real = scores[0]
        prob_fake = scores[1]
        pred_idx = 0 if prob_real > prob_fake else 1
        confidence = max(scores)
        
        output = {
            "predicted": LABEL_MAP[pred_idx],
            "confidence": confidence,
            "prob_real": prob_real,
            "prob_fake": prob_fake,
            "text_preview": text[:220] + "..." if len(text) > 220 else text
        }
        
        if true_label is not None:
            true_str = LABEL_MAP[true_label]
            output.update({
                "true_label": true_str,
                "correct": pred_idx == true_label
            })
        
        return output
    
    except Exception as e:
        return {"error": str(e)}

In [35]:
# Cell 4 – Simple single-text test (edit the text)

custom_text = """
Surajkund swing tragedy revives grim history of ride, ropeway accidents in India.
The collapse of a giant swing at the Surajkund International Crafts Mela in Haryana has once again put the spotlight on safety lapses at amusement parks, fairs and ropeway facilities across India, reviving memories of similar accidents reported over the years.

"""

# Optional: set known label (0 = real, 1 = fake) or None
known_label = 0   # ← change or set to None

result = predict_news(custom_text, known_label)

# ────────────────────────────────────────────────
# Display nicely
# ────────────────────────────────────────────────

from IPython.display import display, Markdown

if "error" in result:
    display(Markdown(f"**Error:** {result['error']}"))
else:
    lines = [
        f"**Predicted:** {result['predicted']}  ",
        f"**Confidence:** {result['confidence']:.4f}  ",
        f"**P(REAL):** {result['prob_real']:.4f}  ",
        f"**P(FAKE):** {result['prob_fake']:.4f}  "
    ]
    
    if "true_label" in result:
        correct_emoji = "✅" if result["correct"] else "❌"
        lines.append(f"**True label:** {result['true_label']} {correct_emoji}")
    
    lines.append("\n**Text preview:**")
    lines.append(f"> {result['text_preview']}")
    
    display(Markdown("\n".join(lines)))

**Predicted:** REAL  
**Confidence:** 0.9677  
**P(REAL):** 0.9677  
**P(FAKE):** 0.0323  
**True label:** REAL ✅

**Text preview:**
> 
Surajkund swing tragedy revives grim history of ride, ropeway accidents in India.
The collapse of a giant swing at the Surajkund International Crafts Mela in Haryana has once again put the spotlight on safety lapses at ...